In [1]:
#格式化书本
from __future__ import division, print_function
%matplotlib inline
import sys
sys.path.insert(0, '..')
import book_format
book_format.set_style()

# 将多维方程转换为一维情况

多维卡尔曼滤波器方程与一维滤波器的方程看起来完全不同。然而，如果我们使用一维的状态和测量，这些方程确实会简化为一维方程。本节将帮助你深入理解卡尔曼滤波器方程的实际含义。虽然阅读本节不是理解本书其余部分所必需的，但我建议你仔细阅读，因为这会使后续内容更容易理解。

以下是预测步骤的多维方程。

$$
\begin{aligned}
\mathbf{\bar{x}} &= \mathbf{F x} + \mathbf{B u} \\
\mathbf{\bar{P}} &= \mathbf{FPF}^\mathsf{T} + \mathbf Q
\end{aligned}
$$

对于一维问题，状态 $\mathbf x$ 只有一个变量，因此它是一个 $1\times 1$ 矩阵。我们的运动 $\mathbf{u}$ 也是一个 $1\times 1$ 矩阵。因此，$\mathbf{F}$ 和 $\mathbf B$ 也必须是 $1\times 1$ 矩阵。这意味着它们都是标量，我们可以写成

$$\bar{x} = Fx + Bu$$

这里的变量不是粗体，表示它们不是矩阵或向量。

我们的状态转移很简单——下一个状态与当前状态相同，所以 $F=1$。运动转移也是如此，所以 $B=1$。因此我们有

$$x = x + u$$

这等价于上一章的高斯方程

$$ \mu = \mu_1+\mu_2$$

希望总体过程已经清楚了，接下来我会加快速度。我们有

$$\mathbf{\bar{P}} = \mathbf{FPF}^\mathsf{T} + \mathbf Q$$

同样，由于我们的状态只有一个变量，$\mathbf P$ 和 $\mathbf Q$ 也必须是 $1\times 1$ 矩阵，我们可以将其视为标量，得到

$$\bar{P} = FPF^\mathsf{T} + Q$$

我们已知 $F=1$。标量的转置就是其本身，所以 $F^\mathsf{T} = 1$。这给出

$$\bar{P} = P + Q$$

这等价于高斯方程

$$\sigma^2 = \sigma_1^2 + \sigma_2^2$$

这证明了在维度为 1 的情况下，多维预测方程执行的数学运算与一维方程相同。

以下是更新步骤的方程：

$$
\begin{aligned}
\mathbf{K}&= \mathbf{\bar{P}H}^\mathsf{T} (\mathbf{H\bar{P}H}^\mathsf{T} + \mathbf R)^{-1} \\
\textbf{y} &= \mathbf z - \mathbf{H \bar{x}}\\
\mathbf x&=\mathbf{\bar{x}} +\mathbf{K\textbf{y}} \\
\mathbf P&= (\mathbf{I}-\mathbf{KH})\mathbf{\bar{P}}
\end{aligned}
$$

如上所述，所有矩阵都变成了标量。$H$ 定义了我们如何从位置转换为测量值。两者都是位置，因此不需要转换，即 $H=1$。让我们将已知值代入并一步转换为标量。1x1 矩阵的逆是其值的倒数，因此我们将矩阵求逆转换为除法。

$$
\begin{aligned}
K &=\frac{\bar{P}}{\bar{P} + R} \\
y &= z - \bar{x}\\
x &=\bar{x}+Ky \\
P &= (1-K)\bar{P}
\end{aligned}
$$

在继续证明之前，我希望你看看这些方程，认识到它们实现的是多么简单的概念。残差 $y$ 不过是测量值减去预测值。增益 $K$ 根据我们对上次预测的确定程度与对测量值的确定程度的比较进行缩放。我们基于 $x$ 的旧值加上缩放后的残差值来选择新的状态 $x$。最后，我们根据对测量值的确定程度来更新不确定性。从算法角度来看，这应该与我们在上一章中所做的完全一致。

让我们完成代数证明。回顾一下更新步骤的一维方程：

$$
\begin{aligned}
\mu &=\frac{\sigma_1^2 \mu_2 + \sigma_2^2 \mu_1} {\sigma_1^2 + \sigma_2^2}, \\
\sigma^2 &= \frac{1}{\frac{1}{\sigma_1^2} + \frac{1}{\sigma_2^2}}
\end{aligned}
$$

这里我们令 $\mu_1$ 为状态 $x$，$\mu_2$ 为测量值 $z$。因此 $\sigma_1^2$ 是状态不确定性 $P$，$\sigma_2^2$ 是测量噪声 $R$。让我们代入。

$$\begin{aligned} \mu &= \frac{Pz + Rx}{P+R} \\
\sigma^2 &= \frac{1}{\frac{1}{P} + \frac{1}{R}}
\end{aligned}$$

我先处理 $\mu$。多维情况下的对应方程是

$$
\begin{aligned}
x &= x + Ky \\
&= x + \frac{P}{P+R}(z-x) \\
&= \frac{P+R}{P+R}x + \frac{Pz - Px}{P+R} \\
&= \frac{Px + Rx + Pz - Px}{P+R} \\
&= \frac{Pz + Rx}{P+R}
\end{aligned}
$$

现在来看 $\sigma^2$。多维情况下的对应方程是

$$ 
\begin{aligned}
P &= (1-K)P \\
&= (1-\frac{P}{P+R})P \\
&= (\frac{P+R}{P+R}-\frac{P}{P+R})P \\
&= (\frac{P+R-P}{P+R})P \\
&= \frac{RP}{P+R}\\
&= \frac{1}{\frac{P+R}{RP}}\\
&= \frac{1}{\frac{R}{RP} + \frac{P}{RP}} \\
&= \frac{1}{\frac{1}{P} + \frac{1}{R}}
\quad\blacksquare
\end{aligned}
$$

我们已经证明了，当只有一个状态变量时，多维方程等价于一维方程。最后我要指出一个小问题——我之前对 $H=1$ 和 $F=1$ 的断言过于简化了。一般来说这并不成立。例如，数字温度计可能以电压形式提供测量值，我们需要将其转换为温度，这时就使用 $H$ 进行转换。我略去了这个问题以使解释尽可能简洁明了。将这个推广加入上述方程并重新推导代数是完全直接的，结果依然相同。\\\ 